# Chapter 6: Algorithms for Mixed Strategy Nash Equilibria, with Gambit

This notebook accompanies Chapter 6 of *An Introduction to Computational Game Theory*.
Each algorithm of the chapter is shown twice:

* a **glass-box** version: a short Python implementation that prints the intermediate
  steps the book tabulates (candidate supports, Lemke–Howson paths, polytope vertices);
* the **Gambit** version that one would use in practice.

| Book | Gambit (`pygambit.nash`) |
|---|---|
| Support enumeration (Algorithms 1 and 2) | `enumpoly_solve` (any number of players), `possible_nash_supports` |
| Upper envelope method | (no dedicated method; drawn here) |
| Lemke–Howson | `lcp_solve` |
| EnumMatch / lrsNash | `enummixed_solve` (optionally calling `lrsnash`) |
| Clique | `enummixed_solve(..., cliques=True)` |
| Beyond two players: path following | `gnm_solve`, `ipa_solve`, `hp_solve`, `logit_solve`, `simpdiv_solve` |

Strategies are labelled as in the book: player 1's strategies are $1, \dots, m$ and
player 2's are $m+1, \dots, m+n$.

In [ ]:
import itertools
import time
from fractions import Fraction as F

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cgtbook import (
    P2_COLOUR,
    bimatrix,
    bimatrix_data,
    eq_table,
    load,
    plot_upper_envelope,
    show,
    table_game,
)
from scipy.optimize import linprog

import pygambit as gbt

The games of the chapter, from Gambit's catalog. For the glass-box code we also extract
the payoff matrices as exact fractions: $A$ is player 1's payoff matrix and $B$ player
2's, with rows indexed by player 1's strategies.

In [ ]:
G = {
    # Section 6.1: a degenerate game whose equilibria have supports of sizes 2 and 1
    "unequal supports": load("ch6_unequal_supports_2x3"),
    # Sections 6.1 and 6.5: A = 0, B = identity
    "A=0, B=I": load("ch6_zero_identity_2x2"),
    # the 2x3 running example, with a unique equilibrium
    "2x3 unique": load("ch6_unique_equilibrium_2x3"),
    # Section 6.2
    "2x5": load("ch6_upper_envelope_2x5"),
    # Section 6.2: degenerate variant of the running example (c pays 3/2 to player 2)
    "2x3 degenerate": load("ch6_degenerate_2x3"),
    # Section 6.3: the example with multiple equilibria
    "2x3 multiple": load("ch6_multiple_equilibria_2x3"),
}
GAMES = {name: bimatrix_data(g) for name, g in G.items()}     # (A, B, rows, cols)
G["2x3 unique"]

## 6.1 Support enumeration

### Support enumeration for non-degenerate bimatrix games

In a non-degenerate game the two supports have equal size $k$. For each pair $(I, J)$
with $|I| = |J| = k$, solve the indifference equations (player 1's $x$ on $I$ makes player 2
indifferent on $J$, and vice versa), and keep the solution if it is unique, non-negative,
and satisfies the best response condition.

In [ ]:
def solve_exact(M, b):
    """Solve M z = b in exact arithmetic; return None if M is singular."""
    n = len(M)
    T = [[F(v) for v in row] + [F(bi)] for row, bi in zip(M, b, strict=False)]
    for c in range(n):
        piv = next((r for r in range(c, n) if T[r][c] != 0), None)
        if piv is None:
            return None
        T[c], T[piv] = T[piv], T[c]
        for r in range(n):
            if r != c and T[r][c] != 0:
                f = T[r][c] / T[c][c]
                T[r] = [a - f * p for a, p in zip(T[r], T[c], strict=False)]
    return [T[r][n] / T[r][r] for r in range(n)]


def indifference(M, I, J):  # noqa: E741 (I, J as in the book)
    """Mixed strategy z on I (rows of M) making the columns J of M equally good.
    Unknowns z_i (i in I) and the common payoff w."""
    k = len(I)
    rows = [[M[i][j] for i in I] + [-1] for j in J] + [[1] * k + [0]]
    sol = solve_exact(rows, [0] * len(J) + [1])
    return None if sol is None else (sol[:k], sol[k])


def support_enumeration_nondegenerate(A, B, rows, cols):
    m, n = len(A), len(A[0])
    At = [list(c) for c in zip(*A, strict=False)]                      # A^T
    log = []
    for k in range(1, min(m, n) + 1):
        for I in itertools.combinations(range(m), k):  # noqa: E741
            for J in itertools.combinations(range(n), k):
                name = ("{" + ",".join(rows[i] for i in I) + "}",
                        "{" + ",".join(cols[j] for j in J) + "}")
                sx, sy = indifference(B, I, J), indifference(At, J, I)
                if sx is None or sy is None:
                    log.append((*name, "no unique solution"))
                    continue
                (xI, v), (yJ, u) = sx, sy
                x = [F(0)] * m
                y = [F(0)] * n
                for i, p in zip(I, xI, strict=False):
                    x[i] = p
                for j, p in zip(J, yJ, strict=False):
                    y[j] = p
                if min(x) < 0 or min(y) < 0:
                    log.append((*name, "negative probability"))
                    continue
                Ay = [sum(A[i][j] * y[j] for j in range(n)) for i in range(m)]
                xB = [sum(x[i] * B[i][j] for i in range(m)) for j in range(n)]
                if max(Ay) > u or max(xB) > v:
                    log.append((*name, "not a best response"))
                    continue
                eq = f"x = {tuple(map(str, x))}, y = {tuple(map(str, y))}"
                log.append((*name, "equilibrium " + eq))
    return pd.DataFrame(log, columns=["I", "J", "outcome"])


A, B, r, c = GAMES["2x5"]
log = support_enumeration_nondegenerate(A, B, r, c)
log[log.I.str.count(",") == 1]          # the ten candidate pairs with |I| = |J| = 2

Exactly the three supports $\{e,a\}$, $\{a,b\}$, $\{b,d\}$ give equilibria, as the book
finds with the upper envelope. Gambit's `enumpoly_solve` also enumerates supports, and
reports each one it examines through `event_callback`. It does not assume
non-degeneracy, so it also tries supports of unequal sizes, after discarding supports
that cannot carry an equilibrium (the ones `possible_nash_supports` rules out). Here every
support it examines has both of player 1's strategies:

In [ ]:
events = []
res = gbt.nash.enumpoly_solve(G["2x5"], event_callback=events.append)
print(len(events), "supports examined:",
      [{p: list(e.support[p]) for p in G["2x5"].players} for e in events])
eq_table(res.equilibria)

Note that `enumpoly_solve` works in floating point (it solves polynomial systems in
general). For two-player games `enummixed_solve` (Section 6.4) gives exact answers:

In [ ]:
eq_table(gbt.nash.enummixed_solve(G["2x5"]).equilibria)

### Support enumeration for general bimatrix games

In degenerate games supports need not have equal sizes. Algorithm 2 (`alg:sep`) tries all
non-empty pairs $(I, J)$ in order of $|I| + |J|$ and solves the linear feasibility problem
$\mathit{SEP}(G, I, J)$ for each. Here SEP is solved with `scipy.optimize.linprog`.

In [ ]:
def sep(A, B, I, J):  # noqa: E741
    """Feasibility problem SEP(G, I, J); returns (x, y) or None. Variables x, y, u, v."""
    m, n = len(A), len(A[0])
    nv = m + n + 2
    U, V = m + n, m + n + 1
    A_eq, b_eq, A_ub, b_ub = [], [], [], []
    for i in range(m):                                   # (Ay)_i = u on I, <= u off I
        row = [0] * nv
        for j in range(n):
            row[m + j] = float(A[i][j])
        row[U] = -1
        (A_eq if i in I else A_ub).append(row)
        (b_eq if i in I else b_ub).append(0)
    for j in range(n):                                   # (B^T x)_j = v on J, <= v off J
        row = [0] * nv
        for i in range(m):
            row[i] = float(B[i][j])
        row[V] = -1
        (A_eq if j in J else A_ub).append(row)
        (b_eq if j in J else b_ub).append(0)
    A_eq.append([1 if i in I else 0 for i in range(m)] + [0] * (n + 2))
    b_eq.append(1)
    A_eq.append([0] * m + [1 if j in J else 0 for j in range(n)] + [0, 0])
    b_eq.append(1)
    bounds = ([(0, None) if i in I else (0, 0) for i in range(m)]
              + [(0, None) if j in J else (0, 0) for j in range(n)] + [(None, None)] * 2)
    lp = linprog(np.zeros(nv), A_ub=A_ub or None, b_ub=b_ub or None, A_eq=A_eq, b_eq=b_eq,
                 bounds=bounds, method="highs")
    return (lp.x[:m], lp.x[m:m + n]) if lp.status == 0 else None


def nonempty_subsets(k):
    return [S for t in range(1, k + 1) for S in itertools.combinations(range(k), t)]


def support_enumeration_general(A, B, rows, cols):
    m, n = len(A), len(A[0])
    pairs = sorted(itertools.product(nonempty_subsets(m), nonempty_subsets(n)),
                   key=lambda p: len(p[0]) + len(p[1]))
    out = []
    for call, (I, J) in enumerate(pairs, start=1):  # noqa: E741
        sol = sep(A, B, I, J)
        out.append({"call": call, "|I|+|J|": len(I) + len(J),
                    "I": "{" + ",".join(rows[i] for i in I) + "}",
                    "J": "{" + ",".join(cols[j] for j in J) + "}",
                    "feasible": sol is not None,
                    "x": None if sol is None else np.round(sol[0], 4),
                    "y": None if sol is None else np.round(sol[1], 4)})
    return pd.DataFrame(out).set_index("call")


A, B, r, c = GAMES["unequal supports"]
sep_log = support_enumeration_general(A, B, r, c)
print("candidate pairs:", len(sep_log), " first feasible SEP call:",
      sep_log.index[sep_log.feasible][0])
sep_log[sep_log["|I|+|J|"] <= 3]

As in the book: all six pairs with $|I|+|J| = 2$ fail, and among the nine pairs with
$|I|+|J| = 3$ only $I = \{T,B\}, J = \{l\}$ is feasible, so Algorithm 2 makes between 7
and 15 SEP calls (13 with the order used here) out of 21. The solution found by the LP
solver is one point of the set of equilibria $\{(p\,T + (1-p)\,B,\ l) : p \in [1/3, 2/3]\}$.

**What does Gambit do with this degenerate game?** `enumpoly_solve` examines the
support $(\{T,B\}, \{l\})$, finds that its system of equations is singular (it has a
continuum of solutions), and skips it. Since every equilibrium of this game has this
support, *it reports no equilibria at all*:

In [ ]:
events = []
res = gbt.nash.enumpoly_solve(G["unequal supports"], event_callback=events.append)
players = G["unequal supports"].players
for e in events:
    print(f"{type(e).__name__:32s}", {p: list(e.support[p]) for p in players})
print("equilibria found by enumpoly_solve:", res.equilibria)

`enummixed_solve` (vertex enumeration, Section 6.4) handles degeneracy: it finds the two
extreme equilibria $p = 1/3$ and $p = 2/3$, and with `cliques=True` reports that they
form a single maximal Nash subset, the whole segment (Section 6.5).

In [ ]:
res = gbt.nash.enummixed_solve(G["unequal supports"], cliques=True)
print([show(eq) for eq in res.equilibria])
print("maximal Nash subsets:", [[show(eq) for eq in c] for c in res.cliques])

For the game with $A = 0$ and $B = I$, Algorithm 1 (and `enumpoly_solve`) only finds the
two pure equilibria and misses e.g. $((3/4, 1/4), l)$:

In [ ]:
A, B, r, c = GAMES["A=0, B=I"]
print(support_enumeration_nondegenerate(A, B, r, c))
print("enumpoly:", [show(eq) for eq in gbt.nash.enumpoly_solve(G["A=0, B=I"]).equilibria])

## 6.2 Solving games where one player has only two strategies

### The upper envelope method

Player 2's expected payoffs from each column are lines in prob($B$); the upper
envelope gives player 2's best responses, and its kinks are the only candidates for
player 1's equilibrium strategies.

In [ ]:
def envelope_2xn(name, ax):
    A, B, rows, cols = GAMES[name]
    res = gbt.nash.enummixed_solve(G[name])
    eq_pB = sorted({eq["1"]["B"] for eq in res.equilibria})
    lines = [(B[0][j], B[1][j]) for j in range(len(cols))]
    plot_upper_envelope(lines, "prob(B)", "payoff to player 2", cols, P2_COLOUR, ax=ax,
                        marks=eq_pB)
    ax.set_title(f"{name}: equilibria at prob(B) = " + ", ".join(str(p) for p in eq_pB),
                 fontsize=10)


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, name in zip(axes, ["2x3 unique", "2x5", "2x3 degenerate"], strict=False):
    envelope_2xn(name, ax)
plt.tight_layout()
plt.show()

* **2x3 unique**: kinks at $1/3$ ($l,c$) and $2/3$ ($c,r$); only $1/3$ supports an
  equilibrium, $x = (2/3, 1/3)$, $y = (1/2, 1/2, 0)$.
* **2x5**: kinks at $1/2$, $3/5$, $3/4$, each giving one of the three equilibria.
* **2x3 degenerate**: all three lines meet at $1/2$. This point has three best responses
  although its support has size two, so the game is degenerate.

### How can degeneracy arise?

In the degenerate game the equilibria form a segment, which Gambit reports through its
two extreme points:

In [ ]:
res = gbt.nash.enummixed_solve(G["2x3 degenerate"], cliques=True)
print("extreme equilibria:", [show(eq) for eq in res.equilibria])
print("maximal Nash subsets:", [[show(eq) for eq in c] for c in res.cliques])

That is $x = (1/2, 1/2)$ with $y$ anywhere on the segment from $(1/2, 1/2, 0)$ to
$(2/3, 0, 1/3)$, as in the book.

## 6.3 The Lemke–Howson algorithm

### A glass-box implementation

We use the best response polytopes of Section 6.4,
$P = \{\hat x \ge 0 : B^\top \hat x \le 1\}$ and $Q = \{\hat y \ge 0 : A\hat y \le 1\}$,
whose vertices carry the same labels as the vertices of the (extended) best response
diagrams $\tilde X$ and $\tilde Y$ of Section 6.3. Labels $1,\dots,m$ are player 1's
strategies, $m+1,\dots,m+n$ player 2's. A vertex $\hat x$ of $P$ has label $i$ if
$\hat x_i = 0$ and label $m+j$ if $(B^\top \hat x)_j = 1$; similarly for $Q$.

Lemke–Howson then works purely with labels: drop the missing label $k$, move along an
edge to the adjacent vertex, pick up a new label; if it is a duplicate, drop it in the
other polytope, and so on until $k$ is picked up.

In [ ]:
def vertices_with_labels(M, label_offset):
    """Vertices of {z >= 0 : M^T z <= 1} (M is m x n, z in R^m), with their labels.
    Labels of z_i = 0 are 1..m (+offset); labels of tight (M^T z)_j = 1 are the others."""
    m, n = len(M), len(M[0])
    # constraint list: (coefficients, rhs, label)
    cons = [([F(int(i == k)) for k in range(m)], F(0), i + 1) for i in range(m)]
    cons += [([F(M[i][j]) for i in range(m)], F(1), m + j + 1) for j in range(n)]
    verts = {}
    for tight in itertools.combinations(range(len(cons)), m):
        z = solve_exact([cons[t][0] for t in tight], [cons[t][1] for t in tight])
        if z is None or min(z) < 0:
            continue
        if any(sum(a * zi for a, zi in zip(co, z, strict=False)) > rhs for co, rhs, _ in cons[m:]):
            continue
        labels = frozenset(lab for co, rhs, lab in cons
                           if sum(a * zi for a, zi in zip(co, z, strict=False)) == rhs)
        verts[tuple(z)] = frozenset((lab + label_offset - 1) % (m + n) + 1 for lab in labels)
    return verts


def best_response_polytopes(A, B):
    """Vertices of P (for x) and Q (for y) with labels 1..m+n, as in the book.
    Payoffs are shifted to be positive if necessary (this does not change equilibria)."""
    A = [[F(a) for a in row] for row in A]
    B = [[F(b) for b in row] for row in B]
    # P and Q are polytopes if A, B >= 0, A has no zero column and B no zero row
    # (assumption eq:ABpos); otherwise add a constant to all payoffs.

    def shift(M):
        return [[v + 1 - min(min(r) for r in M) for v in row] for row in M]
    if min(min(r) for r in A) < 0 or any(all(v == 0 for v in col) for col in zip(*A)):  # noqa: B905
        A = shift(A)
    if min(min(r) for r in B) < 0 or any(all(v == 0 for v in row) for row in B):
        B = shift(B)
    m, _n = len(A), len(A[0])
    P = vertices_with_labels(B, 0)                          # x-hat: labels 1..m then m+1..m+n
    At = [list(col) for col in zip(*A, strict=False)]
    Q = vertices_with_labels(At, m)                         # y-hat: labels m+1..m+n then 1..m
    return P, Q


def normalise(z):
    s = sum(z)
    return tuple(v / s for v in z) if s else tuple(z)


def lemke_howson(A, B, k, start=None):
    """Follow the LH path for missing label k from `start` (default: the artificial
    equilibrium (0, 0)). Returns the list of steps, in the book's tabular form."""
    P, Q = best_response_polytopes(A, B)
    m, n = len(A), len(A[0])
    for poly, d in ((P, m), (Q, n)):
        if any(len(L) > d for L in poly.values()):
            raise ValueError("the game is degenerate: some vertex has too many labels")
    x, y = start if start else (tuple([F(0)] * m), tuple([F(0)] * n))
    xl, yl = sorted(P[x]), sorted(Q[y])             # positional lists, as in the book's tables
    steps = [(list(yl), list(xl), "start")]
    drop = k
    side = "X" if k in P[x] else "Y"        # k is a label of exactly one of x, y
    while True:
        poly, cur, lab = (P, x, xl) if side == "X" else (Q, y, yl)
        keep = poly[cur] - {drop}
        nxt = [v for v, L in poly.items() if v != cur and keep <= L]
        assert len(nxt) == 1
        new = nxt[0]
        picked = next(iter(poly[new] - keep))
        lab[lab.index(drop)] = picked
        if side == "X":
            x = new
        else:
            y = new
        steps.append((list(yl), list(xl), f"dropped {drop} in {side}, picked up {picked}"))
        if picked == k:
            return steps, (normalise(x), normalise(y))
        drop = picked                       # duplicate label: drop it in the other polytope
        side = "Y" if side == "X" else "X"


def lh_table(A, B, k, start=None):
    steps, (x, y) = lemke_howson(A, B, k, start)
    df = pd.DataFrame([{"labels in Y": " ".join(map(str, yl)),
                        "labels in X": " ".join(map(str, xl)), "step": s}
                       for yl, xl, s in steps])
    return df, x, y

### An example with a unique equilibrium

The path for missing label 4 (Table `tab:uniqueNE-LHpath`):

In [ ]:
A, B, r, c = GAMES["2x3 unique"]
df, x, y = lh_table(A, B, 4)
print("end:", tuple(map(str, x)), tuple(map(str, y)))
df

Path lengths for all missing labels (the book gives 4, 6, 6, 4, 5 pivots):

In [ ]:
{k: len(lemke_howson(A, B, k)[0]) - 1 for k in range(1, 6)}

### An example with multiple equilibria

The path for missing label 2 ends at the black square (Table `tab:multLHpath2`), and the
table of which equilibrium is reached for each missing label (Table `tab:eqfromart`):

In [ ]:
A, B, r, c = GAMES["2x3 multiple"]
SYMBOL = {((F(1), F(0)), (F(0), F(0), F(1))): "white circle",
          ((F(2, 3), F(1, 3)), (F(2, 3), F(0), F(1, 3))): "white square",
          ((F(1, 2), F(1, 2)), (F(1, 2), F(1, 2), F(0))): "black square"}
df, x, y = lh_table(A, B, 2)
print("end:", SYMBOL[(x, y)])
df

In [ ]:
{k: SYMBOL[lemke_howson(A, B, k)[1]] for k in range(1, 6)}

### Finding multiple equilibria: the graph of all Lemke–Howson paths

Starting from every completely labelled pair (the artificial equilibrium and the three
Nash equilibria) and every missing label gives Figure `fig:LHgraph`:

In [ ]:
P, Q = best_response_polytopes(A, B)
complete = [(x, y) for x in P for y in Q if P[x] | Q[y] == set(range(1, 6))]


def name_of(x, y):
    return "artificial" if sum(x) == 0 else SYMBOL[(normalise(x), normalise(y))]


edges = {}
for x, y in complete:
    for k in range(1, 6):
        end = lemke_howson(A, B, k, start=(x, y))[1]
        a, b = name_of(x, y), (SYMBOL[end] if sum(end[0]) else "artificial")
        edges.setdefault(frozenset((a, b)), set()).add(k)
for e, labels in edges.items():
    print(" -- ".join(sorted(e)), "  labels", sorted(labels))

### Lemke–Howson in Gambit

`lcp_solve(game, stop_after=1)` follows one Lemke–Howson path from the artificial
equilibrium, with missing label 1 (player 1's first strategy). Without `stop_after`, it
follows the paths for all labels from the artificial equilibrium and then, recursively,
from every equilibrium found, as in *Finding multiple equilibria*.

In [ ]:
print("missing label 1:", show(gbt.nash.lcp_solve(G["2x3 multiple"], stop_after=1).equilibria[0]))
print("all accessible: ", [show(eq) for eq in gbt.nash.lcp_solve(G["2x3 multiple"]).equilibria])

Gambit has no option for choosing the missing label, but we can choose it by reordering
strategies so that label $k$ comes first (swapping the players if $k$ belongs to
player 2). This reproduces Table `tab:eqfromart` with Gambit:

In [ ]:
def gambit_lh(A, B, k, rows, cols):
    A, B = np.array(A, dtype=object), np.array(B, dtype=object)
    m = len(rows)
    if k <= m:
        order = [k - 1] + [i for i in range(m) if i != k - 1]
        g = bimatrix(A[order], B[order], [rows[i] for i in order], cols)
        eq = gbt.nash.lcp_solve(g, stop_after=1).equilibria[0]
        return tuple(eq["1"][s] for s in rows), tuple(eq["2"][s] for s in cols)
    j = k - 1 - m
    order = [j] + [i for i in range(len(cols)) if i != j]
    g = bimatrix(B.T[order], A.T[order], [cols[i] for i in order], rows)   # players swapped
    eq = gbt.nash.lcp_solve(g, stop_after=1).equilibria[0]
    return tuple(eq["2"][s] for s in rows), tuple(eq["1"][s] for s in cols)


A, B, r, c = GAMES["2x3 multiple"]


def to_frac(t):
    return tuple(F(str(v)) for v in t)


{k: SYMBOL[tuple(map(to_frac, gambit_lh(A, B, k, r, c)))] for k in range(1, 6)}

### Equilibria that Lemke–Howson cannot reach

**Beyond the book.** Gambit's game catalog contains Shapley's (1974) example of a game
with a pair of equilibria that are joined to each other by the paths for *every* missing
label, so no sequence of Lemke–Howson paths from the artificial equilibrium reaches
them. `enummixed_solve` finds all three equilibria; `lcp_solve` finds only one.

In [ ]:
shapley = gbt.catalog.load("journals/other/shapley1974/fig3")
print(shapley.description)
print("enummixed:", [show(eq) for eq in gbt.nash.enummixed_solve(shapley).equilibria])
print("lcp (all accessible):", [show(eq) for eq in gbt.nash.lcp_solve(shapley).equilibria])

In [ ]:
A_s, B_s = [np.vectorize(lambda v: F(str(v)), otypes=[object])(a).tolist()
            for a in shapley.to_arrays()]
P, Q = best_response_polytopes(A_s, B_s)
complete = [(x, y) for x in P for y in Q if P[x] | Q[y] == set(range(1, 7))]
label = {(normalise(x), normalise(y)): f"eq{i}" for i, (x, y) in enumerate(complete) if any(x)}
for x, y in complete:
    ends = [label.get(lemke_howson(A_s, B_s, k, start=(x, y))[1], "artificial")
            for k in range(1, 7)]
    name = label[(normalise(x), normalise(y))] if sum(x) else "artificial"
    print(name, "-> for labels 1..6:", ends)

### Degeneracy

The glass-box implementation assumes non-degeneracy and refuses the degenerate variant
(at prob(B) $= 1/2$ the vertex has labels 3, 4 and 5); Gambit's implementation resolves
degeneracy (lexicographically, see *Degeneracy resolution*) and returns an equilibrium.

In [ ]:
A, B, r, c = GAMES["2x3 degenerate"]
try:
    lemke_howson(A, B, 1)
except ValueError as err:
    print("glass box:", err)
print("Gambit:   ", show(gbt.nash.lcp_solve(G["2x3 degenerate"], stop_after=1).equilibria[0]))

## 6.4 Enumeration of equilibria via best response polytopes

The vertices of $P$ and $Q$ for the running example, with their labels
(Example `ex:enummatch`):

In [ ]:
A, B, r, c = GAMES["2x3 unique"]
P, Q = best_response_polytopes(A, B)


def fmtv(v):
    return "(" + ", ".join(map(str, v)) + ")"


print("P:", {fmtv(v): sorted(L) for v, L in P.items()})
print("Q:", {fmtv(v): sorted(L) for v, L in Q.items()})

**EnumMatch** checks all pairs of non-zero vertices for complete labels; **lrsNash**
instead, for each vertex $\hat x$ of $P$ with labels $K$, enumerates the vertices of the
face $Q(L)$ of $Q$ with the complementary labels $L$.

In [ ]:
pairs = [(x, y) for x in P if any(x) for y in Q if any(y)]
matches = [(x, y) for x, y in pairs if P[x] | Q[y] == set(range(1, 6))]
print(len(pairs), "pairs,", len(matches), "completely labelled:",
      [(fmtv(normalise(x)), fmtv(normalise(y))) for x, y in matches])
for x in P:
    if any(x):
        L = set(range(1, 6)) - P[x]
        face = [fmtv(y) for y in Q if Q[y] >= L]
        x_part = f"x = {fmtv(x)}, labels {sorted(P[x])}"
        print(x_part + f": Q(L) for L = {sorted(L)} has vertices {face}")

Gambit's `enummixed_solve` enumerates the extreme equilibria in this way (it can also
call `lrsnash` from lrslib via `lrsnash_path`, if installed).

How many equilibria can a game have? The game with $A = B = I_m$ (identity matrices)
has $2^m - 1$ equilibria, one for every non-empty support:

In [ ]:
rows = []
for m in range(1, 9):
    eye = np.eye(m, dtype=int)
    g = bimatrix(eye, eye, [f"R{i}" for i in range(m)], [f"c{i}" for i in range(m)])
    t0 = time.perf_counter()
    n_eq = len(gbt.nash.enummixed_solve(g).equilibria)
    rows.append({"m": m, "2^m - 1": 2**m - 1, "enummixed": n_eq,
                 "time (s)": round(time.perf_counter() - t0, 3)})
pd.DataFrame(rows).set_index("m")

**Beyond the book.** von Stengel's (1999) $6 \times 6$ game with 75 equilibria, from
Gambit's catalog. Here chaining Lemke–Howson paths happens to reach all of them:

In [ ]:
vs = gbt.catalog.load("journals/dcg/vonstengel1999/6x6_game_with_75_eq")
print("enummixed:", len(gbt.nash.enummixed_solve(vs).equilibria), "equilibria")
print("lcp_solve (all accessible):", len(gbt.nash.lcp_solve(vs).equilibria), "equilibria")

## 6.5 Degeneracy and non-singleton equilibrium components

For $A = 0$, $B = I$ the four extreme equilibria are $(p, q) = (0,0), (1/2,0), (1/2,1),
(1,1)$ in the book's coordinates $p = $ prob($T$), $q = $ prob($l$), and the
**Clique** algorithm combines them into three maximal Nash subsets
$S_r$, $S_0$ and $S_l$, which together form one component.

In [ ]:
res = gbt.nash.enummixed_solve(G["A=0, B=I"], cliques=True)


def pq(eq):
    return f"(p,q) = ({eq['1']['T']}, {eq['2']['l']})"


print("extreme equilibria:", [pq(eq) for eq in res.equilibria])
for c in res.cliques:
    print("maximal Nash subset with extreme points", [pq(eq) for eq in c])

## Beyond two players (Beyond the book)

For three or more players, equilibria are solutions of polynomial systems and may be
irrational. Gambit offers support enumeration with polynomial equations
(`enumpoly_solve`) and several path-following methods.

The $2\times2\times2$ example of McKelvey and McLennan has 9 equilibria, the maximum
for such games:

In [ ]:
U = {(0, 0, 0): (9, 8, 12), (1, 1, 0): (9, 8, 2), (1, 0, 1): (3, 4, 6), (0, 1, 1): (3, 4, 6)}
arrays = [np.zeros((2, 2, 2), dtype=int) for _ in range(3)]
for prof, pay in U.items():
    for i in range(3):
        arrays[i][prof] = pay[i]
mm = table_game(arrays, [["1", "2"]] * 3, title="McKelvey-McLennan 2x2x2")
res = gbt.nash.enumpoly_solve(mm)
print(len(res.equilibria), "equilibria")
eq_table(res.equilibria)

Path-following methods each find one equilibrium (or, for the global Newton method,
possibly several along one path):

In [ ]:
prior = mm.mixed_strategy_profile([[0.15, 0.85], [0.04, 0.96], [0.28, 0.72]])
print("Herings-Peeters (hp_solve):  ", show(gbt.nash.hp_solve(prior).equilibrium))
print("logit QRE (logit_solve):     ", show(gbt.nash.logit_solve(mm).equilibrium))
print("simplicial subdivision:      ", show(gbt.nash.simpdiv_solve(
    mm.mixed_strategy_profile(rational=True)).equilibrium))
for seed in range(3):
    pert = mm.random_strategy_profile(gen=np.random.default_rng(seed))
    n_eq = len(gbt.nash.gnm_solve(pert).equilibria)
    print(f"global Newton (gnm_solve), seed {seed}: {n_eq} equilibria on the path")

The `event_callback` argument lets us watch a path being followed. The Herings–Peeters
homotopy (mentioned in the chapter) starts from the best responses to a prior; on this
path player 1 switches from strategy 2 to strategy 1 at one value of $t$. The logit quantal response equilibrium path
starts at uniform play ($\lambda = 0$) and approaches an equilibrium as $\lambda \to \infty$.

In [ ]:
hp_events = []
gbt.nash.hp_solve(prior, event_callback=hp_events.append)
logit_events = []
gbt.nash.logit_solve(mm, event_callback=logit_events.append)
logit_path = [e.qre for e in logit_events if type(e).__name__ == "LogitPathEvent"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for player in mm.players:
    axes[0].plot([e.t for e in hp_events], [e.profile[player]["1"] for e in hp_events],
                 ".-", label=f"player {player}")
    axes[1].plot([q.lam for q in logit_path], [q.profile[player]["1"] for q in logit_path],
                 label=f"player {player}")
axes[0].set_xlabel("homotopy parameter t")
axes[0].set_title("Herings-Peeters path", fontsize=10)
axes[1].set_xlabel(r"$\lambda$")
axes[1].set_xscale("log")
axes[1].set_title("logit QRE path", fontsize=10)
for ax in axes:
    ax.set_ylabel("probability of strategy 1")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The three-player Stag Hunt from the Chapter 5 notebook has an irrational equilibrium
($1/\sqrt 2$). Different methods find different equilibria:

In [ ]:
arrays = [np.empty((2, 2, 2), dtype=object) for _ in range(3)]
for s in np.ndindex(2, 2, 2):
    for i in range(3):
        arrays[i][s] = 1 if s[i] == 1 else (2 if s == (0, 0, 0) else 0)
stag3 = table_game(arrays, [["Deer", "Hare"]] * 3, title="Three-player Stag Hunt")
print("enumpoly:", [show(eq) for eq in gbt.nash.enumpoly_solve(stag3).equilibria])
print("logit:   ", show(gbt.nash.logit_solve(stag3).equilibrium))
print("liap:    ", show(gbt.nash.liap_solve(stag3.mixed_strategy_profile()).equilibrium))

## 6.6 The complexity of computing a mixed Nash equilibrium

Table `tab:algsummary`, with the corresponding Gambit methods:

| Method | Output | Gambit |
|---|---|---|
| Support enumeration, non-degenerate | all equilibria | `enumpoly_solve` (skips singular supports, see Section 6.1) |
| Support enumeration with SEP | one; all extreme with equal-sized candidates | (glass box above) |
| Upper envelope, $2\times n$ | all equilibria | (picture above) |
| Lemke–Howson | one equilibrium (more by chaining paths) | `lcp_solve` |
| EnumMatch, lrsNash | all extreme equilibria | `enummixed_solve`, `lrsnash_path=` |
| Clique | all equilibria, as maximal Nash subsets | `enummixed_solve(cliques=True)` |